# Milestone 2.1 — Data Cleaning & Chunking
**Input:** `extracted_papers.jsonl` from Google Drive  
**Output:** `chunked_corpus.jsonl` saved to Google Drive  
**Owner:** M1  

Steps:
1. Mount Google Drive
2. Load `extracted_papers.jsonl`
3. Deduplicate by abstract cosine similarity (threshold > 0.95)
4. Chunk body text into 512-token segments with 64-token overlap
5. Save `chunked_corpus.jsonl`

## Step 0 — Install Dependencies

In [ ]:
!pip install -q sentence-transformers langchain langchain-text-splitters transformers tqdm

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2 — Configure Paths

⚠️ **Update `BASE_DIR` to match your Google Drive folder structure.**

In [ ]:
import os

# ── UPDATE THIS to match your Drive folder ──────────────────────────────────
BASE_DIR = '/content/drive/MyDrive/arxiv-llm-project/data'
# ────────────────────────────────────────────────────────────────────────────

INPUT_FILE  = os.path.join(BASE_DIR, 'processed', 'extracted_papers.jsonl')
OUTPUT_FILE = os.path.join(BASE_DIR, 'processed', 'chunked_corpus.jsonl')

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

print('Input :', INPUT_FILE)
print('Output:', OUTPUT_FILE)
print('Input exists:', os.path.exists(INPUT_FILE))

Input : /content/drive/MyDrive/arxiv-llm-project/data/processed/extracted_papers.jsonl
Output: /content/drive/MyDrive/arxiv-llm-project/data/processed/chunked_corpus.jsonl
Input exists: True


## Step 3 — Load extracted_papers.jsonl

In [ ]:
import json

papers = []
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            papers.append(json.loads(line))

print(f'Loaded {len(papers)} papers')
print('\nSample keys:', list(papers[0].keys()) if papers else 'No papers found')
print('\nFirst paper preview:')
p = papers[0]
print(f'  paper_id : {p.get("paper_id", p.get("id", "N/A"))}')
print(f'  title    : {p.get("title", "N/A")[:80]}')
print(f'  abstract : {str(p.get("abstract", "N/A"))[:120]}...')
print(f'  body_text length: {len(str(p.get("body_text", "")))} chars')

Loaded 5000 papers

Sample keys: ['paper_id', 'title', 'abstract', 'year', 'category', 'url', 'body_text', 'source']

First paper preview:
  paper_id : 2201.00084v1
  title    : Performance Comparison of Deep Learning Architectures for Artifact Removal in Ga
  abstract : Endoscopic images typically contain several artifacts. The artifacts significantly impact image analysis result in compu...
  body_text length: 17538 chars


## Step 4 — Deduplication by Abstract Cosine Similarity

Papers with abstract cosine similarity > 0.95 are considered duplicates. We keep the first occurrence.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

DEDUP_THRESHOLD = 0.95

print('Loading embedding model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract abstracts — fall back gracefully if field is missing
abstracts = [str(p.get('abstract', p.get('title', ''))) for p in papers]

print(f'Embedding {len(abstracts)} abstracts...')
embeddings = embed_model.encode(abstracts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print('\nRunning deduplication...')
keep_indices = []
dropped = 0

for i in tqdm(range(len(papers))):
    if not keep_indices:
        keep_indices.append(i)
        continue
    # Compare against all kept embeddings
    kept_embs = embeddings[keep_indices]          # shape (k, dim)
    sim = cosine_similarity([embeddings[i]], kept_embs)[0]  # shape (k,)
    if sim.max() < DEDUP_THRESHOLD:
        keep_indices.append(i)
    else:
        dropped += 1

deduped_papers = [papers[i] for i in keep_indices]
print(f'\nOriginal : {len(papers)} papers')
print(f'Dropped  : {dropped} duplicates')
print(f'Kept     : {len(deduped_papers)} papers')

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Embedding 5000 abstracts...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]


Running deduplication...


100%|██████████| 5000/5000 [00:21<00:00, 235.00it/s]


Original : 5000 papers
Dropped  : 0 duplicates
Kept     : 5000 papers


## Step 5 — Chunking with RecursiveCharacterTextSplitter

Each paper body is split into 512-token segments with 64-token overlap.  
Chunk IDs follow the format: `{paper_id}_chunk_{n}`

In [10]:
# Filter to target categories only
target_categories = {'cs.LG', 'cs.AI', 'cs.CL', 'cs.CV'}
deduped_papers = [p for p in deduped_papers if p.get('category', '') in target_categories]
print(f'After category filter: {len(deduped_papers)} papers remaining')

# Rest of existing code below...
from langchain_text_splitters import RecursiveCharacterTextSplitter
...
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

# Use LLaMA tokenizer for accurate token counting
# If you don't have access, we fall back to character-based approximation (~4 chars/token)
CHUNK_SIZE_TOKENS   = 512
CHUNK_OVERLAP_TOKENS = 64

try:
    print('Loading LLaMA tokenizer for token-accurate chunking...')
    tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-v0.1')

    def length_function(text):
        return len(tokenizer.encode(text, add_special_tokens=False))

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_TOKENS,
        chunk_overlap=CHUNK_OVERLAP_TOKENS,
        length_function=length_function,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    print('Using LLaMA tokenizer.')

except Exception as e:
    print(f'Tokenizer not available ({e})')
    print('Falling back to character-based splitter (approx 4 chars per token)...')
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE_TOKENS * 4,
        chunk_overlap=CHUNK_OVERLAP_TOKENS * 4,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    print('Using character-based splitter.')

After category filter: 3216 papers remaining
Loading LLaMA tokenizer for token-accurate chunking...
Using LLaMA tokenizer.


In [11]:
all_chunks = []
skipped_no_body = 0

for paper in tqdm(deduped_papers, desc='Chunking papers'):
    paper_id = str(paper.get('paper_id', paper.get('id', 'unknown')))
    body     = str(paper.get('body_text', paper.get('full_text', '')))

    # If no body text, fall back to abstract
    if not body.strip():
        body = str(paper.get('abstract', ''))
        skipped_no_body += 1

    if not body.strip():
        continue  # Skip entirely empty records

    chunks = splitter.split_text(body)

    for n, chunk_text in enumerate(chunks):
        all_chunks.append({
            'chunk_id'  : f'{paper_id}_chunk_{n}',
            'paper_id'  : paper_id,
            'chunk_index': n,
            'total_chunks': len(chunks),
            'title'     : paper.get('title', ''),
            'url'       : paper.get('url', ''),
            'year'      : paper.get('year', ''),
            'category'  : paper.get('category', ''),
            'abstract'  : paper.get('abstract', ''),
            'text'      : chunk_text
        })

print(f'\nTotal chunks created : {len(all_chunks)}')
print(f'Papers using abstract fallback: {skipped_no_body}')
print(f'Avg chunks per paper : {len(all_chunks) / len(deduped_papers):.1f}')

Chunking papers: 100%|██████████| 3216/3216 [06:46<00:00,  7.91it/s]  


Total chunks created : 76443
Papers using abstract fallback: 0
Avg chunks per paper : 23.8


## Step 6 — Sanity Check

In [12]:
import pandas as pd

# Show sample chunk
sample = all_chunks[0]
print('=== Sample Chunk ===')
for k, v in sample.items():
    val = str(v)
    print(f'  {k:15s}: {val[:100]}{"..." if len(val)>100 else ""}')

# Distribution of chunks per paper
chunks_per_paper = {}
for c in all_chunks:
    chunks_per_paper[c['paper_id']] = c['total_chunks']

values = list(chunks_per_paper.values())
print(f'\n=== Chunk Distribution ===')
print(f'  Min chunks/paper : {min(values)}')
print(f'  Max chunks/paper : {max(values)}')
print(f'  Median           : {sorted(values)[len(values)//2]}')

=== Sample Chunk ===
  chunk_id       : 2201.00095v1_chunk_0
  paper_id       : 2201.00095v1
  chunk_index    : 0
  total_chunks   : 39
  title          : Computer Vision Based Parking Optimization System
  url            : http://arxiv.org/abs/2201.00095v1
  year           : 2022
  category       : cs.CV
  abstract       : An improvement in technology is linearly related to time and time-relevant problems. It has been see...
  text           : Computer Vision Based Parking Optimization
System
Siddharth Chandrasekaran, Jeffrey Matthew Reginald...

=== Chunk Distribution ===
  Min chunks/paper : 1
  Max chunks/paper : 665
  Median           : 21


## Step 7 — Save chunked_corpus.jsonl to Google Drive

In [13]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f'Saved {len(all_chunks)} chunks to:')
print(f'  {OUTPUT_FILE}')
print(f'  File size: {file_size_mb:.2f} MB')
print('\n✅ Milestone 2.1 complete — chunked_corpus.jsonl ready for M2 (embedding + FAISS)')

Saved 76443 chunks to:
  /content/drive/MyDrive/arxiv-llm-project/data/processed/chunked_corpus.jsonl
  File size: 208.87 MB

✅ Milestone 2.1 complete — chunked_corpus.jsonl ready for M2 (embedding + FAISS)


## Step 8 — (Optional) Quick Retrieval Smoke Test

Verify that chunks make semantic sense by running a quick similarity search.

In [14]:
# Load a small sample and run a query
SAMPLE_SIZE = min(500, len(all_chunks))
sample_texts = [c['text'] for c in all_chunks[:SAMPLE_SIZE]]

print(f'Embedding {SAMPLE_SIZE} sample chunks for smoke test...')
sample_embs = embed_model.encode(sample_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)

TEST_QUERY = 'attention mechanism in transformer models'
query_emb  = embed_model.encode([TEST_QUERY], normalize_embeddings=True)
sims       = cosine_similarity(query_emb, sample_embs)[0]
top3       = sims.argsort()[-3:][::-1]

print(f'\nTop 3 chunks for query: "{TEST_QUERY}"\n')
for rank, idx in enumerate(top3, 1):
    c = all_chunks[idx]
    print(f'--- Rank {rank} (sim={sims[idx]:.3f}) ---')
    print(f'  chunk_id : {c["chunk_id"]}')
    print(f'  title    : {c["title"][:80]}')
    print(f'  text     : {c["text"][:200]}...')
    print()

Embedding 500 sample chunks for smoke test...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Top 3 chunks for query: "attention mechanism in transformer models"

--- Rank 1 (sim=0.468) ---
  chunk_id : 2201.00168v1_chunk_14
  title    : Self-attention Multi-view Representation Learning with Diversity-promoting Compl
  text     : to 0.0001. For the conﬁguration of three baseline models, we
use max-pooling, mean-pooling, or weighted summation to
take replace of the self-attention mechanism in SAMVDPC,
and set λ in objective fun...

--- Rank 2 (sim=0.453) ---
  chunk_id : 2201.00168v1_chunk_5
  title    : Self-attention Multi-view Representation Learning with Diversity-promoting Compl
  text     : Neural Machine Translation (NMT) [22, 25], and rapidly ap-
plied to numerous application domains and achieved promis-
ing results on several challenging tasks, such as image cap-
tioning [26], and sum...

--- Rank 3 (sim=0.451) ---
  chunk_id : 2201.00199v1_chunk_2
  title    : The GatedTabTransformer. An enhanced deep learning architecture for tabular mode
  text     : Radostin Cholak